In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
network = pypsa.Network(
    "../../resources/network/Historical_2015_reduced_solved.nc"
)

In [ ]:
network 

## 1. Baseline Network Analysis

This notebook analyses the Historical_2015_reduced PyPSA-GB baseline scenario.

The initial objectives are to:

- inspect network topology and generation composition;
- identify binding transmission constraints;
- quantify renewable curtailment;
- examine the relationship between congestion and wind curtailment;
- establish a baseline for later BESS and market-design experiments.

In [ ]:
# Basic model information

print("Scenario:", network.name)
print("Snapshots:", len(network.snapshots))
print("Start:", network.snapshots[0])
print("End:", network.snapshots[-1])

print("\nNetwork components")
print("------------------")
print("Buses:", len(network.buses))
print("Lines:", len(network.lines))
print("Generators:", len(network.generators))
print("Loads:", len(network.loads))
print("Storage units:", len(network.storage_units))
print("Links:", len(network.links))

## 2. Transmission Congestion Analysis

For each transmission line, calculate the maximum observed active power flow and compare it with the modelled line limit.

The purpose is to identify lines that become binding during the optimisation window.

In [ ]:
line_analysis = network.lines[
    ["bus0", "bus1", "s_nom", "s_max_pu"]
].copy()

# Maximum absolute active power flow on each line
line_analysis["max_flow_MW"] = network.lines_t.p0.abs().max()

# Effective modelled line limit
line_analysis["effective_limit_MW"] = (
    line_analysis["s_nom"] * line_analysis["s_max_pu"]
)

# Maximum loading percentage
line_analysis["max_loading_pct"] = (
    100
    * line_analysis["max_flow_MW"]
    / line_analysis["effective_limit_MW"]
)

# Sort from most heavily loaded to least loaded
line_analysis = line_analysis.sort_values(
    "max_loading_pct",
    ascending=False
)

line_analysis.head(15)

In [ ]:
# Calculate hourly loading as a fraction of each line's effective limit
line_loading = network.lines_t.p0.abs().div(
    line_analysis["effective_limit_MW"],
    axis=1
)

# Count how many hours each line is at or above 99% loading
line_analysis["hours_ge_99pct"] = (
    line_loading >= 0.99
).sum()

# Show the most constrained lines
line_analysis[
    [
        "bus0",
        "bus1",
        "effective_limit_MW",
        "max_loading_pct",
        "hours_ge_99pct"
    ]
].head(15)

## 3. Beauly Wind Curtailment Analysis

Beauly contains a large concentration of onshore wind capacity.

This section compares:

- available wind generation;
- actual dispatched wind generation;
- implied wind curtailment;
- transmission congestion on the Beauly–Errochty corridor.

Curtailment is calculated as available renewable generation minus actual dispatched generation.

In [ ]:
beauly_wind = network.generators[
    (network.generators["bus"] == "Beauly")
    & (network.generators["carrier"] == "wind_onshore")
]

beauly_wind.head()

In [ ]:
beauly_wind_capacity = beauly_wind["p_nom"].sum()

print(
    "Beauly onshore wind capacity:",
    round(beauly_wind_capacity, 1),
    "MW"
)

In [ ]:
# Generator names
beauly_wind_names = beauly_wind.index

# Maximum wind generation possible in each hour
wind_available = (
    network.generators_t.p_max_pu[beauly_wind_names]
    .mul(beauly_wind["p_nom"], axis=1)
    .sum(axis=1)
)

# Actual optimised wind dispatch
wind_dispatch = (
    network.generators_t.p[beauly_wind_names]
    .sum(axis=1)
)

# Implied curtailment
wind_curtailment = (
    wind_available - wind_dispatch
).clip(lower=0)

print("Available:", round(wind_available.sum(), 1), "MWh")
print("Dispatched:", round(wind_dispatch.sum(), 1), "MWh")
print("Curtailed:", round(wind_curtailment.sum(), 1), "MWh")

print(
    "Curtailment rate:",
    round(
        100 * wind_curtailment.sum() / wind_available.sum(),
        2
    ),
    "%"
)

### 3.1 Wind Curtailment and Network Congestion

The hourly wind availability and dispatch results are combined with loading on the
Beauly–Errochty transmission corridor.

A line is classified as constrained when its loading is at or above 99% of its
modelled limit.

In [ ]:
# Beauly–Errochty transmission line used for this initial analysis
beauly_line = "1"

# Hourly loading of the line
beauly_line_loading = line_loading[beauly_line] * 100

# Build one analysis dataframe
beauly_analysis = pd.DataFrame({
    "wind_available_MW": wind_available,
    "wind_dispatch_MW": wind_dispatch,
    "wind_curtailment_MW": wind_curtailment,
    "line_loading_pct": beauly_line_loading
})

# Identify constrained hours
beauly_analysis["constrained"] = (
    beauly_analysis["line_loading_pct"] >= 99
)

beauly_analysis.head(10)

In [ ]:
# Compare constrained and unconstrained hours
congestion_comparison = (
    beauly_analysis
    .groupby("constrained")
    .agg(
        hours=("wind_curtailment_MW", "count"),
        avg_available_wind_MW=("wind_available_MW", "mean"),
        avg_dispatch_MW=("wind_dispatch_MW", "mean"),
        avg_curtailment_MW=("wind_curtailment_MW", "mean"),
        total_curtailment_MWh=("wind_curtailment_MW", "sum")
    )
)

congestion_comparison

In [ ]:
## 4. Baseline Wind Curtailment Visualisation

The following figure compares hourly available wind generation with actual dispatched
wind generation at Beauly. The gap between the two represents implied wind curtailment.

In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load solved network
network = pypsa.Network(
    "../../resources/network/Historical_2015_reduced_solved.nc"
)

# Select Beauly onshore wind generators
beauly_wind = network.generators[
    (network.generators["bus"] == "Beauly")
    & (network.generators["carrier"] == "wind_onshore")
]

beauly_wind_names = beauly_wind.index

# Available wind
wind_available = (
    network.generators_t.p_max_pu[beauly_wind_names]
    .mul(beauly_wind["p_nom"], axis=1)
    .sum(axis=1)
)

# Actual wind dispatch
wind_dispatch = (
    network.generators_t.p[beauly_wind_names]
    .sum(axis=1)
)

# Wind curtailment
wind_curtailment = (
    wind_available - wind_dispatch
).clip(lower=0)

# Remove tiny numerical noise
wind_curtailment[wind_curtailment < 1e-6] = 0

# Beauly -> Errochty line
beauly_line = "1"

line_limit = (
    network.lines.at[beauly_line, "s_nom"]
    * network.lines.at[beauly_line, "s_max_pu"]
)

beauly_line_loading = (
    network.lines_t.p0[beauly_line].abs()
    / line_limit
    * 100
)

# Build combined dataframe
beauly_analysis = pd.DataFrame({
    "wind_available_MW": wind_available,
    "wind_dispatch_MW": wind_dispatch,
    "wind_curtailment_MW": wind_curtailment,
    "line_loading_pct": beauly_line_loading
})

beauly_analysis["constrained"] = (
    beauly_analysis["line_loading_pct"] >= 99
)

beauly_analysis.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    beauly_analysis.index,
    beauly_analysis["wind_available_MW"],
    label="Available wind"
)

ax.plot(
    beauly_analysis.index,
    beauly_analysis["wind_dispatch_MW"],
    label="Dispatched wind"
)

ax.fill_between(
    beauly_analysis.index,
    beauly_analysis["wind_dispatch_MW"],
    beauly_analysis["wind_available_MW"],
    alpha=0.3,
    label="Curtailment"
)

ax.set_title("Beauly Onshore Wind Availability and Dispatch")
ax.set_ylabel("Power (MW)")
ax.set_xlabel("Time")
ax.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# List active scenarios
python -c "
import yaml
with open('config/config.yaml') as f:
    config = yaml.safe_load(f)
print('Active scenarios:', config.get('run_scenarios', []))
"